# Iris Classification

**Goal:** predict Iris species from sepal/petal length and width.

This notebook performs EDA, visualizes class separability, compares k-NN, Logistic Regression and Decision Tree, reports evaluation metrics, saves the best model, and demonstrates inference.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
DATA_PATH = Path("data/iris.csv")
MODEL_PATH = Path("models/iris_best_model.joblib")
FEATURES = ["sepal_length","sepal_width","petal_length","petal_width"]
TARGET = "species"

## 1. Load and inspect data

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.dtypes)

## 2. Data quality

In [ ]:
print("Missing values:")
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Class counts:")
display(df[TARGET].value_counts())

## 3. Descriptive statistics

In [ ]:
display(df[FEATURES].describe().T)

## 4. Class distribution

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(data=df, x=TARGET)
plt.title("Iris Species Distribution")
plt.xlabel("Species")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 5. Feature distributions

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(12,8))
for ax, feature in zip(axes.ravel(), FEATURES):
    sns.boxplot(data=df,x=TARGET,y=feature,ax=ax)
    ax.set_title(feature.replace("_"," " ).title())
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 6. Class separability

The pair plot helps inspect whether the three species form distinct clusters. Petal measurements provide particularly strong separation.

In [ ]:
sns.pairplot(df, vars=FEATURES, hue=TARGET, diag_kind="hist", height=2.2)
plt.suptitle("Pairwise Feature Relationships and Class Separability", y=1.02)
plt.show()

## 7. Correlation matrix

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[FEATURES].corr(),annot=True,fmt=".2f",cmap="coolwarm",square=True)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 8. Train/test split

In [ ]:
X=df[FEATURES]
y=df[TARGET]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print("Train:",X_train.shape,"Test:",X_test.shape)

## 9. Train and compare models

In [ ]:
models={
    "k-NN":Pipeline([("scaler",StandardScaler()),("classifier",KNeighborsClassifier(n_neighbors=5))]),
    "Logistic Regression":Pipeline([("scaler",StandardScaler()),("classifier",LogisticRegression(max_iter=1000))]),
    "Decision Tree":DecisionTreeClassifier(random_state=42,max_depth=4)
}
results=[]
predictions={}
fitted_models={}
for name,model in models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    fitted_models[name]=model
    predictions[name]=pred
    results.append({"Model":name,"Accuracy":accuracy_score(y_test,pred),"Precision (weighted)":precision_score(y_test,pred,average="weighted",zero_division=0),"Recall (weighted)":recall_score(y_test,pred,average="weighted",zero_division=0)})
results_df=pd.DataFrame(results).sort_values(["Accuracy","Precision (weighted)","Recall (weighted)"],ascending=False).reset_index(drop=True)
display(results_df.style.format({"Accuracy":"{:.4f}","Precision (weighted)":"{:.4f}","Recall (weighted)":"{:.4f}"}))

## 10. Compare metrics visually

In [ ]:
plot_df=results_df.melt(id_vars="Model",value_vars=["Accuracy","Precision (weighted)","Recall (weighted)"],var_name="Metric",value_name="Score")
plt.figure(figsize=(10,6))
sns.barplot(data=plot_df,x="Model",y="Score",hue="Metric")
plt.ylim(0,1.05)
plt.title("Model Performance Comparison")
plt.tight_layout()
plt.show()

## 11. Confusion matrices

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
labels=sorted(y.unique())
for ax,(name,pred) in zip(axes,predictions.items()):
    cm=confusion_matrix(y_test,pred,labels=labels)
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",cbar=False,xticklabels=labels,yticklabels=labels,ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 12. Precision/recall reports

In [ ]:
for name,pred in predictions.items():
    print("="*70)
    print(name)
    print("="*70)
    print(classification_report(y_test,pred,digits=4))

## 13. Save the best model

In [ ]:
best_model_name=results_df.iloc[0]["Model"]
best_model=fitted_models[best_model_name]
MODEL_PATH.parent.mkdir(parents=True,exist_ok=True)
joblib.dump(best_model,MODEL_PATH)
print("Best model:",best_model_name)
print("Saved:",MODEL_PATH)

## 14. Example inference

In [ ]:
loaded_model=joblib.load(MODEL_PATH)
new_flower=pd.DataFrame([[5.1,3.5,1.4,0.2]],columns=FEATURES)
prediction=loaded_model.predict(new_flower)[0]
display(new_flower)
print("Predicted species:",prediction)

## 15. Conclusion

The workflow covers data validation, EDA, class-separability visualization, three classical classifiers, metric comparison, confusion matrices, model persistence and inference.